## Installation
Gymnasium isn't part of the Python standard library, so it needs to be installed once per environment (computer / virtual environment / notebook server). If you've already installed it, you can skip this cell — running `pip install` again is harmless, it'll just confirm it's already there.


In [2]:
%pip install "gymnasium[classic-control]" matplotlib

Note: you may need to restart the kernel to use updated packages.


## Imports
Same idea as any Python script: we import the libraries we need before using them.

- `gymnasium` — the RL environment toolkit itself
- `numpy` — for arrays and numerical operations (Gymnasium observations come back as NumPy arrays)
- `matplotlib.pyplot` — for plotting our agent's learning progress, and for building our video clips


In [3]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

SEED = 0
np.random.seed(SEED)

## Creating the Environment
We create environments with `gym.make(environment_id)`. The id is just a string Gymnasium looks up in its registry — `"Acrobot-v1"` here. (The `-v1` is a version number; environments occasionally get updated, and the version number tells you exactly which rules apply.)


In [4]:
env = gym.make("Acrobot-v1")
env

<TimeLimit<OrderEnforcing<PassiveEnvChecker<AcrobotEnv<Acrobot-v1>>>>>

In [5]:
observation, info = env.reset(seed=SEED)

print("Observation:", observation)
print("Info:", info)

Observation: [ 0.99962485  0.02738891  0.9989402  -0.04602639 -0.09180529 -0.09669447]
Info: {}


## 5. Understanding the Observation
The printed array has **6 numbers**. Acrobot's documentation tells us what they mean:

| Index | Meaning |
|---|---|
| 0 | cos(theta1) — cosine of the first joint's angle |
| 1 | sin(theta1) — sine of the first joint's angle |
| 2 | cos(theta2) — cosine of the second joint's angle |
| 3 | sin(theta2) — sine of the second joint's angle |
| 4 | angular velocity of joint 1 |
| 5 | angular velocity of joint 2 |

In [6]:
print("Observation Space:", env.observation_space)
print("Shape:", env.observation_space.shape)
print("Lower bounds:", env.observation_space.low)
print("Upper bounds:", env.observation_space.high)

Observation Space: Box([ -1.        -1.        -1.        -1.       -12.566371 -28.274334], [ 1.        1.        1.        1.       12.566371 28.274334], (6,), float32)
Shape: (6,)
Lower bounds: [ -1.        -1.        -1.        -1.       -12.566371 -28.274334]
Upper bounds: [ 1.        1.        1.        1.       12.566371 28.274334]


In [7]:
print("Action space:", env.action_space)
print("Number of actions:", env.action_space.n)

# .sample() picks a uniformly random valid action
print("A random action:", env.action_space.sample())

Action space: Discrete(3)
Number of actions: 3
A random action: 1


In [8]:
observation, info = env.reset(seed=SEED)

action = env.action_space.sample()  # pick a random action
observation, reward, terminated, truncated, info = env.step(action)

print("Action taken:", action)
print("New observation:", observation)
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)

Action taken: 2
New observation: [ 0.99996984 -0.0077642   0.9997182  -0.02373883 -0.25169677  0.31000718]
Reward: -1.0
Terminated: False
Truncated: False


In [9]:
observation, info = env.reset(seed=SEED)
total_reward = 0
steps = 0

while True:
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    steps+= 1

    done = terminated or truncated
    if done:
        break

In [10]:
from IPython.display import HTML
from matplotlib import animation


def collect_random_episode_frames(seed=SEED, max_steps=500):
    """Run one episode with random actions and return the list of rendered frames."""
    render_env = gym.make("Acrobot-v1", render_mode="rgb_array")
    obs, info = render_env.reset(seed=seed)

    frames = [render_env.render()]
    for _ in range(max_steps):
        action = render_env.action_space.sample()
        _, _, terminated, truncated, _ = render_env.step(action)
        frames.append(render_env.render())
        if terminated or truncated:
            break

    render_env.close()
    return frames

def make_animation(frames, title=""):
    """Turn a list of image frames into an inline, playable animation."""
    fig, ax = plt.subplots()
    ax.axis("off")
    if title:
        ax.set_title(title)
    img = ax.imshow(frames[0])

    def update(i):
        img.set_data(frames[i])
        return [img]

    anim = animation.FuncAnimation(
        fig, update, frames=len(frames), interval=40, blit=True
    )
    plt.close(fig)  # prevents a duplicate static image from also being displayed
    return anim

random_frames = collect_random_episode_frames()
print(f"Collected {len(random_frames)} frames from the random-action episode.")

random_anim = make_animation(random_frames, title="Random actions")
HTML(random_anim.to_jshtml())


Collected 501 frames from the random-action episode.


In [11]:
# Number of bins per dimension.
# This grows multiplicatively! With 6 dimensions, 6 bins each gives
# 6**6 ~ 46,656 states
N_BINS = 6

obs_low = env.observation_space.low
obs_high = env.observation_space.high

# Angular velocities are technically unbounded by physics but Gymnasium
# documents a practical range
print("Low: ", obs_low)
print("High: ", obs_high)

# Build one set of bin edges per dimension
bin_edges = [
    np.linspace(obs_low[i], obs_high[i], N_BINS - 1)
    for i in range(len(obs_low))
]

def discretize(observation):
    """Convert a continuous 6-value observation into a tuple of 6 bin indices."""
    return tuple(
        int(np.digitize(observation[i], bin_edges[i]))
        for i in range(len(observation))
    )

# Quick check
obs, info = env.reset(seed=SEED)
print("Raw observation:  ", obs)
print("Discretized state:", discretize(obs))

Low:  [ -1.        -1.        -1.        -1.       -12.566371 -28.274334]
High:  [ 1.        1.        1.        1.       12.566371 28.274334]
Raw observation:   [ 0.99962485  0.02738891  0.9989402  -0.04602639 -0.09180529 -0.09669447]
Discretized state: (4, 3, 4, 2, 2, 2)


In [12]:
Q = {}  # maps discretized_state -> array of Q-values, one per action

def get_q_values(state):
    """Return the Q-values for a state, creating a fresh all-zero row if we 
    haven't seen it before."""
    if state not in Q:
        Q[state] = np.zeroes(env.action_space.n)
    return Q[state]

In [ ]:
def epsilon_greedy_action(state, epsilon):
    if np.random.rand() < epsilon:
        return env.action_space.sample()  # explore
    else:
        return int(np.argmax(get_q_values(state)))  # exploit

## Hyperparameters
Four knobs control how training behaves:
- **alpha** - learning rate: how much each new experience updates our existing estimate
- **gamma** - discount factor: how much we value future rewards vs. immediate ones
- **epsilon** - exploration rate, starting high and decaying over time
- **episodes** - how many practice rounds to run



In [14]:
alpha = 0.1  # learning rate
gamma = 0.99  # discount factor

epsilon = 1.0  # start by exploring 100% of the time
epsilon_min = 0.05  # never fully stop exploring
epsilon_decay = 0.9995  # multiply epsilon by this after every episode

episodes = 5000
max_steps_per_episode = 500 

# Tracking, for plotting later
episode_rewards = []

In [ ]:
for ep in range(episodes):
    obs, info = env.reset()
    state = discretize(obs)
    total_reward = 0

    for t in range(max_steps_per_episode):
        action = epsilon_greedy_action(state, epsilon)

        next_obs, reward, terminated, truncated, info = env.step(action)
        next_state = discretize(next_obs)
        done = terminated = truncated

        # Q-learning update
        best_next_value = np.max(get_q_values(next_state))
        td_target = reward + gamma * best_next_value
        td_error = td_target - get_q_values(state)[action]
        Q[state][action] += alpha * td_error

        state = next_state
        total_reward += reward

        if done:
            break

    episode_rewards.append(total_reward)

    # Decay epsilon, but never below epsilon_min
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if (ep + 1) % 500 == 0:
        recent_avg = np.mean(episode_rewards[-500:])
        print(f"Episode {ep + 1}/{episodes} | epsilon={epsilon:.3f} | avg reward (last 500): {recent_avg:.1f}")